# Verification workflow demo for Paper 1 v207

This notebook demonstrates a **synthetic metadata-style verification workflow** for MCT provenance-credential contribution events. It does not call external APIs, deploy smart contracts, or issue financial tokens. The goal is to show how DOI/ORCID/repository/evidence fields can be separated from non-transferable credential representation.

> Scope note: all records in `data/example_contributions.json` are synthetic. Passing these checks means only that a record is internally well-formed for the demonstration; it does not mean that a real repository, DOI, ORCID, or curator decision has been authenticated.


## 1. Load synthetic contribution events

The example records encode contribution type, contributor identity, research-object metadata, evidence links, verification status, scoring inputs, and locked credential metadata.


In [1]:
from pathlib import Path
import json, re

events = json.loads(Path('data/example_contributions.json').read_text())
len(events)


## 2. Run lightweight metadata checks

The checks below intentionally use simple regular expressions and URL checks. In a real system, each item would be replaced with authenticated calls to ORCID, DOI registration agencies, repositories, journals, or institutional curation systems.


In [2]:
ORCID_RE = re.compile(r'^\\d{4}-\\d{4}-\\d{4}-\\d{3}[0-9X]$')
DOI_RE = re.compile(r'^10\\.[^\\s/]+/.+')

def verify_event(event):
    checks = {
        'has_valid_orcid': bool(ORCID_RE.match(event['contributor'].get('orcid', ''))),
        'has_doi': bool(DOI_RE.match(event['research_object'].get('doi', ''))),
        'has_repository_url': event['research_object'].get('repository_url', '').startswith(('http://', 'https://')),
        'has_evidence_link': len(event.get('evidence', {}).get('links', [])) > 0,
        'is_non_transferable': event['issued_credential'].get('non_transferable') is True,
        'is_locked': event['issued_credential'].get('locked') is True,
    }
    passed = sum(checks.values())
    if event['verification']['status'] == 'curator_verified' and passed == len(checks):
        workflow_status = 'curator_verified_event'
    elif passed >= 5:
        workflow_status = 'metadata_complete_pending_curator'
    else:
        workflow_status = 'incomplete_metadata'
    return {'event_id': event['event_id'], 'contribution_type': event['contribution_type'], **checks, 'checks_passed': passed, 'workflow_status': workflow_status}

verification_rows = [verify_event(e) for e in events]
verification_rows[:2]


## 3. Export reviewer-inspectable verification results

The output table is designed for auditability: each row reports the event identifier and whether each required metadata class is present in the synthetic record.


In [3]:
import csv
out = Path('outputs/verification_results.csv')
out.parent.mkdir(exist_ok=True)
with out.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(verification_rows[0].keys()))
    writer.writeheader()
    writer.writerows(verification_rows)
print(f'Wrote {out}')


## Interpretation and limitations

The workflow separates metadata checks from credential representation. In a real deployment, placeholder checks would be replaced with authenticated ORCID, DOI, repository, curator, journal, and/or institutional verification. The synthetic demonstration supports the manuscript claim that contribution verification can be specified independently from non-transferable credential issuance. It does not provide a production verification service, financial token, or deployed blockchain system.
